In [1]:
import polars as pl
from datetime import timedelta, date, datetime
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import numpy as np
from tqdm import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import itertools

# Анализ таргета

In [3]:
events_dir = "../data/raw/T-bank dataset full/T-ECD-small/dataset/small/marketplace/events"

In [4]:
events_df = (
    pl.scan_parquet(f"{events_dir}/*.pq", include_file_paths="path")
    .with_columns(
        pl.col("path")
          .str.extract(r"(\d+)\.pq", group_index=1)
          .cast(pl.Int32)
          .alias("day")
    )
    .sort("day")
    .drop("path")
)

In [5]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32)])

In [6]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day
duration[μs],u64,str,str,str,str,i32
1082d 17h 39m 51s 751152µs,67838893,"""nfmcg_15106540""","""catalog""","""view""","""android""",1082
1082d 17h 39m 51s 827007µs,89736171,"""nfmcg_12135912""","""u2i""","""view""","""android""",1082
1082d 17h 39m 52s 43343µs,11916495,"""nfmcg_22992567""","""u2i""","""view""","""android""",1082
1082d 17h 39m 52s 87139µs,15515933,"""nfmcg_867469""","""other""","""view""","""android""",1082
1082d 17h 39m 52s 132224µs,49878225,"""nfmcg_6159822""","""u2i""","""view""","""android""",1082


In [7]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")
actions_count.head()

action_type,len
str,u32
"""view""",126147783
"""click""",3772094
"""clickout""",485448
"""like""",41792


Создадим также уменьшенный по масштабу таргет: логарифмированный и квадратичный. Идея: постараться естественным (зависимым от статистики) образом создать таргет, при этом не слишком большим по значению, т.к. огромные редкие таргеты могут заставить модели переобучаться и/или искажать оценки ее качества.

Однако гипотезу нужно будет проверить на практике, потому на текущий момент предлагается 3 варианта: стандартный таргет, логарифмированный и таргет в корне:

In [8]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).alias("target"),
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
    (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    
).drop("len")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""view""",1.034082,0.710044,1.016898
"""click""",34.582149,3.571844,5.880659
"""clickout""",268.714913,5.597366,16.392526
"""like""",3121.341812,8.046339,55.86897


In [9]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])

In [10]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 14h 17m 602023µs,27125169,"""nfmcg_27118048""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 14h 17m 8s 542763µs,33507026,"""nfmcg_28041816""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 14h 17m 13s 595050µs,18430120,"""nfmcg_6563951""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 14h 17m 20s 370572µs,85863381,"""nfmcg_2940528""","""other""","""clickout""","""android""",1082,268.714913,5.597366,16.392526,0,1,0,0
1082d 14h 17m 22s 770835µs,67452006,"""nfmcg_11449610""","""catalog""","""click""","""ios""",1082,34.582149,3.571844,5.880659,0,0,0,1


## Global Temporal Split

In [11]:
def global_temporal_split(
    df: pl.LazyFrame, test_size: int | float = 1, date_column: str = "day"
) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """
    Разделяет датасет на обучающую и тестовую части на основе глобальной временной границы. 1 день между тестовой и обучающей частью игнорируется.

    Args:
        df: Датасет для разделения
        date_column: Имя столбца с датами
        test_size: Количество дней в тестовой части или доля от общего количества дней

    Returns:
        Кортеж из двух датасетов: обучающая и тестовая части
    """
    min_day, max_day = (
        df.select(
            pl.col(date_column).min().alias("min_day"), pl.col(date_column).max().alias("max_day")
        )
        .collect(engine="streaming")
        .row(0)
    )
    days_all = (max_day - min_day) + 1
    if isinstance(test_size, float):
        test_size = int(days_all * test_size)
        if test_size == 0:
            test_size += 1
        cut_day = max_day - test_size
    else:
        cut_day = max_day - test_size

    if cut_day - 1 < min_day or cut_day + 1 > max_day:
        raise ValueError(
            f"Test size is too large. Test size: {test_size}, min day: {min_day}, max day: {max_day}, cut day: {cut_day}"
        )

    train_df = df.filter(pl.col(date_column) < cut_day)
    test_df = df.filter(pl.col(date_column) > cut_day)

    return train_df, test_df
    

In [12]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [13]:
def get_last_k_user_interactions(
    events_df: pl.LazyFrame,
    last_k: int | None = 30,
    date_column: str = "day",
    timestamp_column: str = "timestamp",
    user_column: str = "user_id",
    acceptable_action: list[str] | None = None,
):
    if acceptable_action is None:
        acceptable_action = ["view", "clickout", "like", "click"]
    return (
        events_df.filter(pl.col("action_type").is_in(set(acceptable_action)))
        .group_by(user_column)
        .agg(
            pl.all().sort_by(date_column, timestamp_column).tail(last_k)
            if last_k is not None
            else pl.all().sort_by(date_column, timestamp_column)
        )
    )

In [14]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [15]:
train, test = global_temporal_split(events_df, 0.2)

In [16]:
get_last_k_user_interactions(test, acceptable_action=["click", "like", "clickout"]).collect(engine="streaming")

user_id,timestamp,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
u64,list[duration[μs]],list[str],list[str],list[str],list[str],list[i32],list[f64],list[f64],list[f64],list[i8],list[i8],list[i8],list[i8]
10923694,"[1264d 35m 30s 261176µs, 1264d 8h 31m 6s 559750µs, … 1302d 15h 2m 28s 418044µs]","[""nfmcg_39238"", ""nfmcg_20833797"", … ""nfmcg_1428174""]","[""search"", ""search"", … ""catalog""]","[""click"", ""click"", … ""clickout""]","[""other"", ""other"", … ""other""]","[1264, 1264, … 1302]","[34.582149, 34.582149, … 268.714913]","[3.571844, 3.571844, … 5.597366]","[5.880659, 5.880659, … 16.392526]","[0, 0, … 0]","[0, 0, … 1]","[0, 0, … 0]","[1, 1, … 0]"
40961037,"[1283d 21h 45m 56s 526361µs, 1284d 14h 15m 30s 190625µs]","[""nfmcg_13647154"", ""nfmcg_11655282""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""android"", ""android""]","[1283, 1284]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
80268126,"[1302d 2h 3m 45s 391259µs, 1306d 19h 40m 23s 507484µs]","[""nfmcg_11756859"", ""nfmcg_12609992""]","[""other"", ""other""]","[""clickout"", ""clickout""]","[""other"", ""android""]","[1302, 1306]","[268.714913, 268.714913]","[5.597366, 5.597366]","[16.392526, 16.392526]","[0, 0]","[1, 1]","[0, 0]","[0, 0]"
50211779,"[1289d 2h 6m 3s 562706µs, 1289d 3h 30m 10s 138624µs, 1290d 8m 25s 663439µs]","[""nfmcg_2995901"", ""nfmcg_7997336"", ""nfmcg_3969882""]","[""u2i"", ""u2i"", ""u2i""]","[""click"", ""click"", ""click""]","[""android"", ""android"", ""ios""]","[1289, 1289, 1290]","[34.582149, 34.582149, 34.582149]","[3.571844, 3.571844, 3.571844]","[5.880659, 5.880659, 5.880659]","[0, 0, 0]","[0, 0, 0]","[0, 0, 0]","[1, 1, 1]"
48631355,"[1279d 23h 28m 29s 136229µs, 1280d 15h 42m 36s 859194µs]","[""nfmcg_19353812"", ""nfmcg_26409985""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""android"", ""android""]","[1279, 1280]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
8474676,"[1271d 6h 24m 30s 190663µs, 1271d 14h 34m 48s 356668µs]","[""nfmcg_17242917"", ""nfmcg_7326721""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""ios"", ""ios""]","[1271, 1271]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
15045866,"[1282d 3h 44m 13s 958129µs, 1282d 11h 56m 55s 766929µs]","[""nfmcg_611356"", ""nfmcg_22178359""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""android"", ""ios""]","[1282, 1282]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
49641637,[1272d 13h 38m 44s 555863µs],"[""nfmcg_22479815""]","[""u2i""]","[""click""]","[""android""]",[1272],[34.582149],[3.571844],[5.880659],[0],[0],[0],[1]


In [17]:
def split_cold_start(train_df: pl.LazyFrame, test_df: pl.LazyFrame, user_col: str = "user_id"):
    """
    Split test data into cold-start and non-cold-start subsets by users.

    Parameters
    ----------
    train_df : pl.LazyFrame
        Training data. Used to determine which users are already known to the model.
    test_df : pl.LazyFrame
        Test data that will be split into cold-start and non-cold-start parts.
    user_col : str, optional
        Name of the column containing user identifiers, by default "user_id".

    Returns
    -------
    tuple[pl.LazyFrame, pl.LazyFrame]
        A tuple of two LazyFrames:

        - first element : test subset with cold-start users
          (users present in `test_df` but not in `train_df`);
        - second element : test subset with non-cold-start users
          (users present both in `train_df` and `test_df`).
    """
    cold_start_users = test_df.select(pl.col(user_col).unique()).join(
        train_df.select(pl.col(user_col).unique()), on=user_col, how="anti"
    )
    return test_df.join(cold_start_users, on=user_col), test_df.join(
        cold_start_users, on=user_col, how="anti"
    )

In [18]:
def ndcg_at_k(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
):
    """
    Computes user-based NDCG@k for graded relevance in a recommendation setting.

    Parameters
    ----------
    user_based_df : pl.DataFrame
        Dataframe with user data. Each row must contain user and its lists with: truth
        ground items, their relevancy estimation and model prediction score.
    relevancy_col : str
        Column name contains list of relevancy estimations ground
        truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
    true_items_col : str
        Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
        of these items must be set in `relevancy_col` respectively. 
    predicted_items_col : str
        Columns name with predicted items (pl.List[str]). Must be set in order matches
        `predicted_score_col`.
    predicted_score_col : str
        Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
        Used to sort predictions in descending order.
    top_k : int, optional
        Top k elements to calculate `@k` metric.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``user_id`` : user identifier;
        - ``ndcg`` : NDCG@k for current user.

    Notes
    -----
    For each user, the function:
    1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
    2. Joins predicted items with their ground-truth relevancies.
    3. Computes DCG@k using the order induced by the model (sorting by score).
    4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
    5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
    """
    user_ids = []
    ndcgs = []
    for row in user_based_df.iter_rows(named=True):
        true_items = pl.DataFrame(
            {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
        )
        true_items = true_items.group_by("truth_items").agg(
            pl.col("relevancy").max()
        )  # Берем максимальную релевантность для товара
        predictions = (
            pl.DataFrame(
                {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
            )
            .join(
                true_items,
                left_on="predicted_items",
                right_on="truth_items",
                coalesce=True,
                how="left",
            )
            .fill_null(0)
        )
        idcg = (
            predictions.select("relevancy")
            .sort("relevancy", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        dcg = (
            predictions.select("score", "relevancy")
            .sort("score", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        user_ids.append(row["user_id"])
        ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
    return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

In [19]:
user_based_df = pl.DataFrame({
    "user_id": ["u1", "u2"],
    "true_items": [
        ["A", "B", "C"],   # истиные товары
        ["A", "B", "C"],
    ],
    "relevancy": [
        [3.0, 2.0, 1.0],   # u1: A=3, B=2, C=1
        [3.0, 2.0, 1.0],   # u2: A=3, B=2, C=1
    ],
    "predicted_items": [
        ["A", "B", "C"],   # u1: идеальное ранжирование
        ["C", "B", "A"],   # u2: худшее (перевёрнутое)
    ],
    "predicted_scores": [
        [0.9, 0.8, 0.7],   # по убыванию A>B>C
        [0.9, 0.8, 0.7],   # по убыванию C>B>A
    ],
})

In [20]:
ndcg_at_k(
    user_based_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=3,
)

user_id,ndcg
str,f64
"""u1""",1.0
"""u2""",0.789998


## Добавляем метрики Precision@k и Recall@k

Пусть:
- $Rel\_u$ — множество релевантных объектов для пользователя $u$ в тесте.
- $Rec\_u@k$ — множество рекомендованных объектов в топ‑k для пользователя $u$.

Тогда для одного пользователя:


$$
Precision@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{k} = \frac{ hits}{k}
$$
то есть, сколько релевантных из рекомендованных.

$$
Recall@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{|Rel\_u|} = \frac{hits}{Rel\_u}
$$
то есть, сколько релевантных найдено из всех релевантных.

Далее берем среднее по всем пользователям:
$$
Precision@k = \frac{1}{N} \sum_{u=1}^{N} Precision@k(u)
$$
$$
Recall@k = \frac{1}{N} \sum_{u=1}^{N} Recall@k(u)
$$



Precision@k показывает качество самого списка рекомендаций. Она говорит о том, сколько "мусора" мы подсовываем пользователю.

Recall@k показывает охват интересов. Она говорит: "Из всего многообразия товаров, которые этот пользователь мог бы полюбить, какую долю мы реально успели ему показать в нашем коротком топе?".

В рекомендательных системах (особенно когда k маленькое, например 10 или 20) Recall обычно важнее, чем Precision. Precision тоже важна, чтобы не раздражать пользователя нерелевантным мусором, но часто мы готовы пожертвовать точностью (показать пару лишних товаров), лишь бы не упустить то, что пользователь реально ищет (сохранить Recall).

In [21]:
def calculate_metrics(df, k):
    """
    Расчет Precision@k и Recall@k
    df: датафрейм с колонками predicted_items и true_items
    k: количество рекомендаций (TOPK)
    """
    # Обрезаем список предсказаний до k элементов
    top_k_preds = pl.col("predicted_items").list.head(k)
    
    # Ищем пересечение обрезанного списка с правдой
    hits_expr = top_k_preds.list.set_intersection(pl.col("true_items")).list.len()
    
    # Вычисляем метрики одной командой select
    metrics = df.select(
        # Precision = (кол-во попаданий / k), берем среднее по всем юзерам
        (hits_expr / k).mean().alias('precision'),
        
        # Recall = (кол-во попаданий / длину реального списка), берем среднее
        (hits_expr / pl.col('true_items').list.len()).mean().alias('recall')
    )
    
    precision_val = metrics['precision'].item()
    recall_val = metrics['recall'].item()

    return precision_val, recall_val

## Добавляем метрики RMSE и MAE

In [22]:
def calculate_regression_metrics_vectorized(
    test_data,
    user_features: np.ndarray,
    item_features: np.ndarray,
    user_to_idx: dict,
    item_to_idx: dict,
    target_col: str = "log_target",
) -> dict:
    
    # Фильтруем только известные пары user-item
    test_filtered = test_data.filter(
        pl.col("user_id").is_in(list(user_to_idx.keys())) &
        pl.col("item_id").is_in(list(item_to_idx.keys()))
    )
    
    # Получаем индексы
    user_ids = test_filtered["user_id"].to_list()
    item_ids = test_filtered["item_id"].to_list()
    true_scores = test_filtered[target_col].to_numpy()
    
    user_idxs = np.array([user_to_idx[uid] for uid in user_ids])
    item_idxs = np.array([item_to_idx[iid] for iid in item_ids])
    
    # Вычисление скоров
    # pred_score[i] = user_features[user_idx[i]] @ item_features[:, item_idx[i]]
    pred_scores = np.sum(
        user_features[user_idxs] * item_features[:, item_idxs].T, 
        axis=1
    )
    
    # Метрики
    errors = true_scores - pred_scores
    rmse = np.sqrt(np.mean(errors ** 2))
    mae = np.mean(np.abs(errors))
    
    return {
        "rmse": rmse,
        "mae": mae,
        "n_samples": len(true_scores),
    }

Комментарий к коду выше:

* Результат user_predictions — это вектор, где в каждой ячейке лежит число, показывающее "силу связи" (значимость) между этим пользователем и конкретным товаром. Чем больше число, тем сильнее мы хотим порекомендовать этот товар.

* ind_sorted это массив индексов товаров. Индексы идут в порядке убывания скора товара. То есть ind_sorted[0] — это индекс самого подходящего товара, ind_sorted[1] — второго по крутости и т.д.

* top_k_item_ids это оригинальные ID товаров

* top_k_scores это те значимости (скоры), которые мы посчитали (user_predictions). Мы достаем их из большого массива user_predictions по нужным индексам.

Замечание: в функции выше мы исключаем товары, с которыми пользователь взаимодействовал. Скорее всего это не совсем правильный подход. Может быть, стоит исключать только те, которые были просмотренные, а остальные оставлять. 

# Будем реализовывать Truncated SVD

https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html#sklearn.decomposition.TruncatedSVD

>TruncatedSVD implements a variant of singular value decomposition (SVD) that only computes the largest singular values, where is a user-specified parameter.

Источник: https://scikit-learn.org/stable/modules/decomposition.html#lsa

В sklearn.TruncatedSVD используются приближённые алгоритмы SVD. Ищутся k наибольших сингулярных значений и соответствующие им сингулярные векторы, а вклад всех остальных считается пренебрежимо малым и отбрасывается.

Если оставить algorithm='randomized', используется рандомизированный алгоритм Халко (https://arxiv.org/abs/0909.4061) для поиска низкорангового приближения матрицы по первым k сингулярным компонентам. Мы заранее задаём количество компонент n_components = k.

`TruncatedSVD` минимизирует норму Фробениуса разности между исходной матрицей и её приближением.

# Описание бейзлайна

**Будем реализовывать Truncated SVD c количеством латентных факторов LATENT_FACTORS = 30 и потом выбирать TOP_K = 15 рекомендаций**

1) Из `train` выбираются столбцы `user_id`, `item_id`, `target` / `sqrt_target` / `log_target` и из них составляется DataFrame `train_data`
2) Фильтруем датафрейм так, чтобы остались только пользователи и айтемы с количеством взаимодейтсвий >= 5
3) Переводим оригинальные id юзеров и айтемов в простые числовые индексы
4) Строим матрицу взаимодействий `R_train_sparse` (User-Item матрица), где по строкам индексы пользователей, по столбцам индексы айтемов, в ячейках - значение таргета. Храним ее в формате хранения csr.
> Если пары (user, item) не было в train_data_indexed, она не попадает в списки индексов и значений, передаваемых в конструктор csr_matrix. В разреженных матрицах (scipy.sparse.csr_matrix) любые координаты, для которых явно не указано значение, по определению считаются равными нулю.

5) Для восстановления тех значений, которые были нулями будем использовать `TruncatedSVD` из `sklearn.decomposition`.

>`TruncatedSVD` (усеченное сингулярное разложение) берет нашу разреженную матрицу взаимодействий (где строки — пользователи, столбцы —    товары) и приближенно раскладывает её в произведение трех матриц:
>
>$$ R \approx U \times \Sigma \times V^T $$
>
>Здесь:
>1. $U$ - показывает отношение пользователей к скрытым факторам.
>2. $\Sigma$ - диагональная матрица с сингулярными числами. Эти числа показывают важность каждого скрытого фактора. Чем больше число, тем больше этот фактор влияет на итоговый выбор.
>3. $V^T$ - показывает, насколько каждый скрытый фактор выражен в конкретном товаре.
>
>
>В `sklearn`:
>
>* матрица `User Features` ($U \cdot \Sigma$) это результат вызова `svd_model.fit_transform(R)`. Каждый пользователь описывается вектором из $k$ чисел (в коде это `LATENT_FACTORS`=20).
>   
>* матрица `Item Features` ($V^T$) это `svd_model.components_`. Каждый товар описывается вектором из $k$ чисел.
>
>Эти 'скрытые факторы' (latent factors) — это неявные характеристики, выявленные алгоритмом.
>
>Когда мы перемножаем полученные матрицы обратно, мы получаем новую плотную матрицу, которая:
>
>* в местах, где были данные старается быть максимально похожей на исходные значения (`target`, `log_target`,`sqrt_target`).
>* в местах, где были нули заполняет их не нулями, а вещественными числами (предсказаниями). Изначально модель стремится предсказать здесь нули (минимизируя ошибку - норму Фробениуса), но из-за использования малого числа компонент ($k$) она вынуждена обобщать паттерны.
>
>Мы сортируем эти предсказанные числа для всех товаров, которые пользователь еще не видел, и рекомендуем те, где предсказанное значение максимально.

In [23]:
def run_svd_experiment(train_data, test_data, target_col='log_target', latent_factors=20, top_k=15):
    print(f'🚀 Запуск эксперимента: Target={target_col}, Factors={latent_factors}, TopK={top_k}')

    # Подготовка данных для построения матрицы взаимодействий
    # Выбираем столбцы 'user_id', 'item_id', target_col из train, это будет датафрейм train_data
    train_data = train_data.select(['user_id', 'item_id', target_col]).collect(engine = 'streaming')

    # Оставляем пользователей и айтемы с количеством взаимодействий >= 5. Это нужно для экономии памяти. 
    # В результате количество строк в train_data уменьшилось с 101598458 до 98851489
    MIN_INTERACTIONS = 5

    user_counts = train_data.group_by('user_id').agg(pl.len().alias('user_count'))
    item_counts = train_data.group_by('item_id').agg(pl.len().alias('item_count'))

    valid_users = user_counts.filter(pl.col('user_count') >= MIN_INTERACTIONS).select('user_id')
    valid_items = item_counts.filter(pl.col('item_count') >= MIN_INTERACTIONS).select('item_id')

    train_data = (
        train_data
        .join(valid_users, on='user_id', how="inner")
        .join(valid_items, on='item_id', how="inner")
    )
    print(f'Взаимодействий после фильтрации: {train_data.height}')

    # Далее нужно построить матрицу взаимодействий и преобразовать ее в csr-формат
    # Создаем таблицу уникальных id с присвоенным индексом (idx)
    user_ids_unique = train_data.select(pl.col('user_id')).unique().with_row_index('user_idx') 
    item_ids_unique = train_data.select(pl.col('item_id')).unique().with_row_index('item_idx')

    # Сохраняем размеры матрицы
    num_users = user_ids_unique.height
    num_items = item_ids_unique.height
    print(f"Размерность матрицы: {num_users} пользователей x {num_items} предметов")

    # Преобразуем в словарь {'user_idx': [0, 1, ..], 'user_id': [34917205, 2230613, ...]}
    user_mapping_dict = user_ids_unique.to_dict(as_series=False)
    item_mapping_dict = item_ids_unique.to_dict(as_series=False)

    # Преобразование оригинальных id в простые числовые индексы (нужно для матрицы взаимодействий)
    user_to_idx = dict(zip(user_mapping_dict['user_id'], user_mapping_dict['user_idx']))
    item_to_idx = dict(zip(item_mapping_dict['item_id'], item_mapping_dict['item_idx']))

    # Обратный переход 
    idx_to_item = {idx: item_id for item_id, idx in item_to_idx.items()}
    idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}

    # Присоединяем созданные индексы (user_idx, item_idx)
    train_data_indexed = train_data.join(user_ids_unique, on='user_id', how='left').join(item_ids_unique, on='item_id', how='left')

    # Подготовка данных для CSR матрицы
    user_indices = train_data_indexed['user_idx'].to_numpy()
    item_indices = train_data_indexed['item_idx'].to_numpy()
    relevance_values = train_data_indexed[target_col].to_numpy()

    # Создание User-Item матрицы в формате хранения csr
    R_train_sparse = csr_matrix(
        (relevance_values, (user_indices, item_indices)),
        shape=(num_users, num_items)
    )

    print(f'Размер обучающей CSR матрицы: {R_train_sparse.shape}')

    # Обучение TruncatedSVD
    svd_model = TruncatedSVD(n_components=latent_factors, random_state=42)

    # user_features = U * S
    user_features = svd_model.fit_transform(R_train_sparse)
    # item_features = V^T
    item_features = svd_model.components_

    def get_top_k_recommendations(user_idx: int, k: int = 15):
        # user_features - матрица эмбеддингов пользователей (U*S), полученная через fit_transform
        # Берем вектор конкретного пользователя
        user_feature_vector = user_features[user_idx] 
    
        # Считаем скоры (предсказания) для всех товаров
        # Мы умножаем вектор пользователя на матрицу всех товаров
        user_predictions = user_feature_vector @ item_features 

        # Достаем историю взаимодействий пользователя из разреженной обучающей матрицы
        # .toarray().flatten() превращает разреженную строку в обычный плоский массив
        user_train_row = R_train_sparse.getrow(user_idx).toarray().flatten()
        # Фильтруем уже известные взаимодействия:
        # Присваиваем им score = -inf, чтобы при сортировке они гарантированно оказались в конце
        user_predictions[user_train_row > 0] = -np.inf
    
        # argpartition находит индексы k самых больших элементов, но не сортирует их
        # это сделано для уменьшения времени исполнения, до этого было 12-13 часов (когда argsort использовалось), сейчас уменьшилось почти до 1 часа
        ind_unsorted = np.argpartition(user_predictions, -k)[-k:]
    
        # Сортируем только эти k элементов
        ind_sorted = ind_unsorted[np.argsort(user_predictions[ind_unsorted])[::-1]]
    
        top_k_item_ids = [idx_to_item[idx] for idx in ind_sorted]
        top_k_scores = [user_predictions[idx] for idx in ind_sorted]
    
        return top_k_item_ids, top_k_scores

    # Оценка на тесте
    test_data = test_data.select(['user_id', 'item_id', target_col]).collect(engine='streaming')

    # Оставляем только тех пользователей, которых SVD знает
    known_users = set(user_to_idx.keys())
    test_data_filtered = test_data.filter(pl.col('user_id').is_in(known_users))
    print(f"Пользователей в тесте после фильтрации: {test_data_filtered.select('user_id').n_unique()}")

    user_test_truth = (
        test_data_filtered.group_by('user_id')
        .agg(
            pl.col('item_id').alias('true_items'),
            pl.col(target_col).alias('relevancy'),
        )
    )

    # Параллельная генерация рекомендаций
    NUM_WORKERS = os.cpu_count() if os.cpu_count() is not None else 4
    test_user_ids = user_test_truth['user_id'].to_numpy()
    total_users = len(test_user_ids)
    predicted_items_list = [None] * total_users
    predicted_scores_list = [None] * total_users

    # Функция-обертка для ProcessPoolExecutor
    def process_user(user_id, index):
        """Генерирует рекомендации и возвращает их вместе с исходным индексом."""
        user_idx = user_to_idx[user_id] 
        top_items, top_scores = get_top_k_recommendations(user_idx, k=top_k)
        return index, top_items, top_scores

    print(f"Начало параллельной генерации рекомендаций на {NUM_WORKERS} ядрах...")

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    
        futures = [
            executor.submit(process_user, user_id, i)
            for i, user_id in enumerate(test_user_ids)
        ]
    
        # as_completed и tqdm для отслеживания прогресса
        for future in tqdm(as_completed(futures), total=total_users):
            index, top_items, top_scores = future.result()
            predicted_items_list[index] = top_items
            predicted_scores_list[index] = top_scores

    print("\nПараллельная генерация рекомендаций завершена.")

    # Формирование итогового DataFrame
    prediction_df = pl.DataFrame({
        'user_id': test_user_ids,
        'predicted_items': predicted_items_list,
        'predicted_scores': predicted_scores_list,
    })

    evaluation_df = user_test_truth.join(prediction_df, on='user_id', how='inner')

    # Вычисляем средний NDCG@TOP_K
    ndcg_results = ndcg_at_k(
        evaluation_df,
        relevancy_col='relevancy',
        true_items_col='true_items',
        predicted_items_col='predicted_items',
        predicted_score_col='predicted_scores',
        top_k=top_k,
    )
    mean_ndcg = ndcg_results['ndcg'].mean()

    # Вычисляем Precision@TOP_K и Recall@TOP_K
    precision, recall = calculate_metrics(evaluation_df, k=top_k)

    metrics = calculate_regression_metrics_vectorized(
        test_data_filtered,  # тестовый датафрейм
        user_features,
        item_features,
        user_to_idx,
        item_to_idx,
        target_col=target_col
    )

    rmse = metrics['rmse']
    mae = metrics['mae']

    return {
        'target': target_col,
        'latent_factors': latent_factors,
        'top_k': top_k,
        'ndcg': mean_ndcg,    
        'precision': precision, 
        'recall': recall,       
        'rmse': rmse,           
        'mae': mae             
    }

In [24]:
# Запуск всех экспериментов
experiment_configs = [
    # Базовые сравнения таргетов
    ('log_target', 20, 15),
    ('sqrt_target', 20, 15),
    ('target', 20, 15),
    
    # Дополнительные проверки для log_target 
    ('log_target', 50, 20),
    ('log_target', 30, 15) 
]

results = []

for t_col, factors, k in experiment_configs:
    print(f"Запускаю: {t_col}, factors={factors}, k={k}...")

    res = run_svd_experiment(train, test, target_col=t_col, latent_factors=factors, top_k=k)
    results.append(res)

final_df = pl.DataFrame(results)
print(final_df)

Запускаю: log_target, factors=20, k=15...
🚀 Запуск эксперимента: Target=log_target, Factors=20, TopK=15
Взаимодействий после фильтрации: 98851489
Размерность матрицы: 1231601 пользователей x 577509 предметов
Размер обучающей CSR матрицы: (1231601, 577509)
Пользователей в тесте после фильтрации: 635751
Начало параллельной генерации рекомендаций на 8 ядрах...


100%|██████████████████████████████████| 635751/635751 [54:52<00:00, 193.08it/s]



Параллельная генерация рекомендаций завершена.
Запускаю: sqrt_target, factors=20, k=15...
🚀 Запуск эксперимента: Target=sqrt_target, Factors=20, TopK=15
Взаимодействий после фильтрации: 98851489
Размерность матрицы: 1231601 пользователей x 577509 предметов
Размер обучающей CSR матрицы: (1231601, 577509)
Пользователей в тесте после фильтрации: 635751
Начало параллельной генерации рекомендаций на 8 ядрах...


100%|██████████████████████████████████| 635751/635751 [51:58<00:00, 203.84it/s]



Параллельная генерация рекомендаций завершена.
Запускаю: target, factors=20, k=15...
🚀 Запуск эксперимента: Target=target, Factors=20, TopK=15
Взаимодействий после фильтрации: 98851489
Размерность матрицы: 1231601 пользователей x 577509 предметов
Размер обучающей CSR матрицы: (1231601, 577509)
Пользователей в тесте после фильтрации: 635751
Начало параллельной генерации рекомендаций на 8 ядрах...


...


Параллельная генерация рекомендаций завершена.
Запускаю: log_target, factors=50, k=20...
🚀 Запуск эксперимента: Target=log_target, Factors=50, TopK=20
Взаимодействий после фильтрации: 98851489
Размерность матрицы: 1231601 пользователей x 577509 предметов
Размер обучающей CSR матрицы: (1231601, 577509)
Пользователей в тесте после фильтрации: 635751
Начало параллельной генерации рекомендаций на 8 ядрах...


100%|████████████████████████████████| 635751/635751 [1:23:02<00:00, 127.59it/s]



Параллельная генерация рекомендаций завершена.
Запускаю: log_target, factors=30, k=15...
🚀 Запуск эксперимента: Target=log_target, Factors=30, TopK=15
Взаимодействий после фильтрации: 98851489
Размерность матрицы: 1231601 пользователей x 577509 предметов
Размер обучающей CSR матрицы: (1231601, 577509)
Пользователей в тесте после фильтрации: 635751
Начало параллельной генерации рекомендаций на 8 ядрах...


100%|██████████████████████████████████| 635751/635751 [58:47<00:00, 180.24it/s]



Параллельная генерация рекомендаций завершена.
shape: (5, 8)
┌─────────────┬────────────────┬───────┬──────────┬───────────┬──────────┬──────────┬──────────┐
│ target      ┆ latent_factors ┆ top_k ┆ ndcg     ┆ precision ┆ recall   ┆ rmse     ┆ mae      │
│ ---         ┆ ---            ┆ ---   ┆ ---      ┆ ---       ┆ ---      ┆ ---      ┆ ---      │
│ str         ┆ i64            ┆ i64   ┆ f64      ┆ f64       ┆ f64      ┆ f64      ┆ f64      │
╞═════════════╪════════════════╪═══════╪══════════╪═══════════╪══════════╪══════════╪══════════╡
│ log_target  ┆ 20             ┆ 15    ┆ 0.135736 ┆ 0.026741  ┆ 0.026168 ┆ 3.111453 ┆ 0.994588 │
│ sqrt_target ┆ 20             ┆ 15    ┆ 0.134087 ┆ 0.027076  ┆ 0.026715 ┆ 4.90131  ┆ 1.498261 │
│ target      ┆ 20             ┆ 15    ┆ 0.084701 ┆ 0.017216  ┆ 0.016236 ┆ 61.31082 ┆ 4.426243 │
│ log_target  ┆ 50             ┆ 20    ┆ 0.123482 ┆ 0.020234  ┆ 0.024746 ┆ 3.172747 ┆ 1.00984  │
│ log_target  ┆ 30             ┆ 15    ┆ 0.131542 ┆ 0.025399  ┆ 0

## Выводы

В результате сравнения разных таргетов при LATENT_FACTORS = 20 и TOP_K = 15 наилучшие значения ранжирующих метрик (NDCG, Precision@k, Recall@k) получены при использовании log_target по сравнению с sqrt_target и сырым target (см. таблицу ниже).

Лучший вариант среди всех проведённых экспериментов — конфигурация с log_target, LATENT_FACTORS = 20 и TOP_K = 15: она даёт максимально высокий NDCG и наилучший баланс Precision@k и Recall@k.

При увеличении числа латентных факторов до 30 NDCG падает до 0.13, Precision и Recall также ухудшаются. Это указывает на то, что усложнение модели не приводит к улучшению качества рекомендаций и, вероятно, начинает подхватывать больше шума.

Конфигурация log_target, LATENT_FACTORS = 50 и TOP_K = 20 даёт ещё более низкие значения NDCG и Precision@k. Это ожидаемо: рост числа факторов повышает риск переобучения, а увеличение k приводит к тому, что в топ‑список попадает больше нерелевантных объектов.

Метрики регрессии для лог‑таргета (RMSE: 3.11, MAE: 1.00 на ~18.9 млн тестовых пар при LATENT_FACTORS = 30, TOP_K = 15) показывают, что SVD лишь грубо аппроксимирует численные значения log_target. Это ожидаемо: TruncatedSVD оптимизирует приближение всей матрицы взаимодействий и не нацелен на точное восстановление таргета. При LATENT_FACTORS = 50 и TOP_K = 20 RMSE (3.17) и MAE (1.01) становятся немного выше, при этом ранжирующие метрики снижаются, что подтверждает, что улучшение по RMSE/MAE не связано с улучшением качества рекомендаций.